# C1 · Collinearity: unstable coefficients, stable predictions

**Type**: Break the model · **Level**: ●●○ independent · **Time**: about 35 minutes · **core path ★**

## Objective

Build two nearly identical features and show with the bootstrap that their OLS coefficients are unstable
(and negatively correlated), while their sum and the predictions stay stable. Then stabilize everything with Ridge.

## Dataset

Synthetic, generated by `generate(n, seed)`: `x2` is `x1` plus a tiny noise (correlation 0.999), `x3` is
independent, and `y = 2·x₁ + 2·x₂ + x₃ + noise`. `X, y` has 80 rows; `X_new, y_new` has 2000 rows of never-seen data.

## Educational purpose

Tell apart **two uses** of a linear model that multicollinearity treats in opposite ways: predicting (it does not suffer)
and interpreting the individual coefficients (it is wrecked by it). And understand *why*: the coefficients compensate each other.

## Task

1. Write `vif(X)` (2-dimensional numpy array): for each column, regress that column on the others with `LinearRegression`, compute the R² and return `1 / (1 - R²)`. Save `vif_values = vif(X)`.
2. With `rng = np.random.default_rng(0)` do 300 bootstrap resamplings of the rows. On each one train `LinearRegression()` and `Ridge(alpha=5)`, saving the coefficients in `coef_ols` and `coef_ridge` (300 × 3 matrices).
3. Compute `std_ols` and `std_ridge` (standard deviation of each coefficient over the 300 fits, `ddof=1`).
4. Compute `std_sum_ols`, the standard deviation of `coef_ols[:, 0] + coef_ols[:, 1]`, and `corr_b1_b2`, the correlation between the two coefficients over the 300 fits.
5. Train OLS and Ridge on the full data and compute `rmse_ols_new` and `rmse_ridge_new` on `X_new, y_new`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression, Ridge


def generate(n, seed):
    rng = np.random.default_rng(seed)
    x1 = rng.normal(size=n)
    x2 = x1 + rng.normal(scale=0.05, size=n)         # almost a copy of x1
    x3 = rng.normal(size=n)                          # independent of the others
    X = np.column_stack([x1, x2, x3])
    y = 2 * x1 + 2 * x2 + 1 * x3 + rng.normal(size=n)
    return X, y


X, y = generate(80, seed=0)
X_new, y_new = generate(2000, seed=1)
print("correlation x1-x2:", round(np.corrcoef(X[:, 0], X[:, 1])[0, 1], 4))

## Your work

In [ ]:
def vif(X):
    ...

vif_values = vif(X)
vif_values.round(1)

In [ ]:
rng = np.random.default_rng(0)
B = 300
coef_ols = np.empty((B, 3))
coef_ridge = np.empty((B, 3))
for b in range(B):
    ...

In [ ]:
std_ols = ...
std_ridge = ...
std_sum_ols = ...
corr_b1_b2 = ...
print("std OLS  :", std_ols.round(3))
print("std Ridge:", std_ridge.round(3))

In [ ]:
rmse_ols_new = ...
rmse_ridge_new = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Bootstrap: `idx = rng.integers(0, len(y), len(y))`, then `LinearRegression().fit(X[idx], y[idx]).coef_`. For the VIF of a column j: `others = np.delete(X, j, axis=1)` and `LinearRegression().fit(others, X[:, j]).score(others, X[:, j])`.'):
    check(vif_values[0] > 50 and vif_values[1] > 50, "x1 and x2 have a huge VIF",
          "the VIF of x1 and x2 should be in the hundreds")
    check(vif_values[2] < 2, "x3 has a VIF close to 1 (it is independent)", "the VIF of x3 should be close to 1")
    check(coef_ols.shape == (300, 3) and coef_ridge.shape == (300, 3), "300 fits per model",
          "coef_ols and coef_ridge must be 300 x 3 matrices")
    check(std_ols[0] > 5 * std_ridge[0] and std_ols[1] > 5 * std_ridge[1],
          "Ridge makes the coefficients of x1 and x2 much more stable",
          "the standard deviation of x1 and x2 with OLS should be at least 5 times the one with Ridge")
    check(std_sum_ols < std_ols[0] / 5,
          "the SUM of the two coefficients is stable even with OLS",
          "the standard deviation of b1 + b2 should be much smaller than that of b1")
    check(corr_b1_b2 < -0.95, "the two coefficients are almost perfectly anti-correlated",
          "over the 300 fits b1 and b2 should have a correlation close to -1")
    check(abs(rmse_ols_new - rmse_ridge_new) / rmse_ols_new < 0.05,
          "the predictions on new data are equivalent", "OLS and Ridge should predict almost the same way")
done()

---
The commented solution is in `solutions/C_break_the_model/C1_collinearity_unstable_coefficients.ipynb`. Open it after you pass the check, or after you have really tried.